# 🚗 Treinamento YOLO11m para Detecção de Tráfego
**Google Colab - GPU Gratuita**

Este notebook treina um modelo YOLO11m para detecção de veículos em vídeos de tráfego rodoviário.

## Como usar:
1. Ative a GPU: **Menu → Runtime → Change runtime type → GPU → Save**
2. Execute cada célula em ordem
3. Baixe o modelo treinado ao final

**Tempo estimado:** 30-60 minutos para 50 épocas com GPU.
Sem GPU: pode levar várias horas ou não completar.


In [ ]:
# ============================================================================
# PASSO 1: Instalar dependências
# ============================================================================
!pip install -q ultralytics opencv-python numpy
print("✅ Dependências instaladas")

In [ ]:
# ============================================================================
# PASSO 2: Baixar dataset do GitHub Release
# ============================================================================
import os

dataset_url = 'https://github.com/LanzaDev/traffic_analysis/releases/download/v1.0/dataset.zip'
print('📥 Baixando dataset do GitHub...')
!wget -q '{dataset_url}' -O dataset.zip

import os
if os.path.exists('dataset.zip'):
    print(f'✅ Dataset baixado: {os.path.getsize("dataset.zip") / 1024 / 1024:.1f} MB')
    !unzip -q dataset.zip
    print('✅ Dataset extraído')
else:
    print('⚠️ Download falhou. Faça upload manual das pastas:')
    print('   assets/train/images/')
    print('   assets/train/labels/')
    print('   assets/val/images/')
    print('   assets/val/labels/')

In [ ]:
# ============================================================================
# PASSO 4: Criar arquivo data.yaml
# ============================================================================
yaml_content = '''train: dataset/train/images
val: dataset/val/images
nc: 6
names:
  0: person
  1: car
  2: motorcycle
  3: bus
  4: truck
  5: articulated_truck'''

with open('data.yaml', 'w') as f:
    f.write(yaml_content)

print('✅ Arquivo data.yaml criado:')
print(yaml_content)

In [ ]:
# ============================================================================
# PASSO 5: Treinar o modelo
# ============================================================================
import torch
from ultralytics import YOLO

device = '0' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    print('⚠️ AVISO: GPU não detectada. Treinando em CPU (será mais lento).')
else:
    print(f'✅ GPU detectada: {torch.cuda.get_device_name(0)}')

model = YOLO('yolo11m.pt')

results = model.train(
    data='data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=device,
    project='runs/train',
    name='traffic_analysis',
    exist_ok=True,
    patience=20,
    save=True,
    verbose=True
)

print('\n✅ Treinamento concluído!')
print(f"📁 Modelo salvo em: runs/train/traffic_analysis/weights/best.pt")

In [ ]:
# ============================================================================
# PASSO 6: Validar o modelo treinado
# ============================================================================
from ultralytics import YOLO

model = YOLO('runs/train/traffic_analysis/weights/best.pt')

metrics = model.val(
    data='data.yaml',
    imgsz=640,
    batch=16,
    device=device
)

print('\n✅ Validação concluída!')
print(f"   mAP50: {metrics.box.map50:.4f}")
print(f"   mAP50-95: {metrics.box.map:.4f}")
print(f"   Precisão: {metrics.box.mp:.4f}")
print(f"   Recall: {metrics.box.mr:.4f}")

In [ ]:
# ============================================================================
# PASSO 7: Baixar o modelo treinado
# ============================================================================
from google.colab import files

model_path = 'runs/train/traffic_analysis/weights/best.pt'

if os.path.exists(model_path):
    print(f"📥 Baixando modelo: {model_path}")
    files.download(model_path)
    print('✅ Download iniciado!')
else:
    print('❌ Modelo não encontrado. Verifique se o treinamento foi concluído.')

# 📋 Como usar o modelo treinado no seu projeto

1. Baixe o arquivo `best.pt` que foi gerado acima
2. Substitua o arquivo `models/yolo11m.pt` no seu projeto pelo novo modelo
3. Execute normalmente:
   ```bat
   run.bat --mode vehicles --source assets/seu_video.mp4 --no-show
   ```

---
## 📚 Dicas para melhorar o resultado:
- Use pelo menos 100-200 imagens anotadas para treino
- Variedade de ângulos, iluminação e clima
- Anote apenas as classes que você precisa (car, truck, bus, motorcycle, person)
- Use `--conf 0.25` para detecção padrão ou `--conf 0.40` para detecção mais conservadora

---
🏁 **Fim do notebook**